# AllLegs — Parallel PPO (multi-environment experience collection)

## Where is the bottleneck?

In `AllLegs_def.ipynb` training is sequential:

```
for each episode:
    rollout_train()   ← 800 steps × (MuJoCo 5×step ≈ 2.5 ms + NN ≈ 0.1 ms)
                      ≈ 2.1 s / episode   ← BOTTLENECK
    if buf full:
        update_ppo()  ← fast, GPU/CPU, ≈ 0.5 s for 10 epochs
```

MuJoCo is single-threaded C — it dominates wall-clock time.  
The neural network is tiny (24→64→8) and negligible.

## Solution: N parallel MuJoCo workers

```
main process (CPU)                subprocess × N_WORKERS
──────────────────                ─────────────────────────────────────────
broadcast weights ──────────────► worker 0: own MjModel → rollout → results
                  ──────────────► worker 1: own MjModel → rollout → results
                  ──────────────► worker 2: own MjModel → rollout → results
                  ──────────────► worker 3: own MjModel → rollout → results
                  ◄──────────────  all results arrive in parallel
concatenate into PPOBuffer
PPO update (when buffer full)
```

**Expected speedup:** ~N_WORKERS × (limited by memory bandwidth + process overhead)

**macOS note:** MuJoCo is not fork-safe. We use `spawn` context explicitly.

In [ ]:
# ── Write the worker module to disk ──────────────────────────────────────────
# The worker runs in a SUBPROCESS, so all its code must live in an importable
# Python file — Jupyter cell definitions are not visible to child processes.

_WORKER_CODE = '''
"""
_parallel_worker.py
Subprocess worker for parallel PPO experience collection.
Receives current policy weights + config, runs one episode, returns transitions.
"""
import os, math
import numpy as np
import torch
import torch.nn as nn
import mujoco


def _worker(worker_id: int, weights_numpy: dict, cfg: dict) -> dict:
    """
    Collect one episode of experience in an independent MuJoCo instance.

    Parameters
    ----------
    worker_id     : int  — used for seeding / debugging
    weights_numpy : dict[str, ndarray] — current policy weights as numpy arrays
    cfg           : dict — all hyperparameters (see build_cfg() in the notebook)

    Returns
    -------
    dict with lists: states, actions, log_probs, rewards, values, dones
         and scalars: last_value, dist, tot_r
    """
    # ── Unpack config ────────────────────────────────────────────────────────
    STATE_DIM    = cfg["STATE_DIM"]
    ACTION_DIM   = cfg["ACTION_DIM"]
    HIDDEN_DIM   = cfg["HIDDEN_DIM"]
    LOG_STD_INIT = cfg["LOG_STD_INIT"]
    LOG_STD_MIN  = cfg["LOG_STD_MIN"]
    LOG_STD_MAX  = cfg["LOG_STD_MAX"]

    SUBSTEPS     = cfg["SUBSTEPS"]
    SETTLE_TIME  = cfg["SETTLE_TIME"]
    EPISODE_DUR  = cfg["EPISODE_DUR"]
    XML_PATH     = cfg["XML_PATH"]

    HIP_INIT     = cfg["HIP_INIT"]
    KNEE_INIT    = cfg["KNEE_INIT"]
    DELTA_HIP    = cfg["DELTA_HIP"]
    DELTA_KNEE   = cfg["DELTA_KNEE"]
    MAX_DDELTA   = cfg["MAX_DDELTA"]
    DELTA_LIMIT  = cfg["DELTA_LIMIT"]

    TARGET_VX    = cfg["TARGET_VX"]
    VX_TRACK_W   = cfg["VX_TRACK_W"]
    VX_TOL       = cfg["VX_TOL"]
    ALIVE_BONUS  = cfg["ALIVE_BONUS"]
    JVEL_W       = cfg["JVEL_W"]
    JEXC_W       = cfg["JEXC_W"]
    ACTION_REG_W = cfg["ACTION_REG_W"]
    DELTA_REG_W  = cfg["DELTA_REG_W"]
    ALIGN_W      = cfg["ALIGN_W"]
    YAW_W        = cfg["YAW_W"]

    OFFSETS  = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
    JOINT_LO = np.array([HIP_INIT - DELTA_HIP,  KNEE_INIT - DELTA_KNEE] * 4, dtype=np.float32)
    JOINT_HI = np.array([HIP_INIT + DELTA_HIP,  KNEE_INIT + DELTA_KNEE] * 4, dtype=np.float32)

    # ── Each subprocess gets its own MuJoCo instance ─────────────────────────
    # MuJoCo model/data are not serialisable — must be created fresh per process.
    mjmodel = mujoco.MjModel.from_xml_path(XML_PATH)
    mjdata  = mujoco.MjData(mjmodel)
    CTRL_DT = mjmodel.opt.timestep * SUBSTEPS

    # ── Reconstruct ActorCritic from transmitted weights ──────────────────────
    # We define the class inline so the worker file is fully self-contained.
    class ActorCritic(nn.Module):
        def __init__(self):
            super().__init__()
            self.pi = nn.Sequential(
                nn.Linear(STATE_DIM, HIDDEN_DIM), nn.ReLU(),
                nn.Linear(HIDDEN_DIM, ACTION_DIM),
            )
            self.log_std = nn.Parameter(torch.full((ACTION_DIM,), LOG_STD_INIT))
            self.vf = nn.Sequential(
                nn.Linear(STATE_DIM, 256), nn.ReLU(),
                nn.Linear(256, 256),       nn.ReLU(),
                nn.Linear(256, 1),
            )

        def get_action(self, s):
            mean    = self.pi(s)
            log_std = self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
            std     = log_std.exp()
            raw     = mean + std * torch.randn_like(std)
            action  = torch.tanh(raw)
            log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                        - log_std - 0.5 * math.log(2 * math.pi))
            log_prob -= torch.log(1 - action.pow(2) + 1e-6)
            log_prob  = log_prob.sum(-1)
            value = self.vf(s).squeeze(-1)
            return action, log_prob, value

    ac = ActorCritic()
    # Convert numpy arrays back to tensors and load
    state_dict = {k: torch.from_numpy(v) for k, v in weights_numpy.items()}
    ac.load_state_dict(state_dict)
    ac.eval()

    # ── Simulation helpers ────────────────────────────────────────────────────
    def orientation():
        qw, qx, qy, qz = mjdata.qpos[3:7]
        roll  = math.atan2(2*(qw*qx + qy*qz), 1 - 2*(qx**2 + qy**2))
        pitch = math.asin(float(np.clip(2*(qw*qy - qz*qx), -1, 1)))
        yaw   = math.atan2(2*(qw*qz + qx*qy), 1 - 2*(qy**2 + qz**2))
        return roll, pitch, yaw

    def build_state(delta):
        return np.concatenate([
            mjdata.qpos[7:15].astype(np.float32),
            mjdata.qvel[6:14].astype(np.float32),
            delta.astype(np.float32),
        ])

    def apply_ddelta(ctrl, delta, ddelta):
        delta_new = np.clip(delta + ddelta * MAX_DDELTA, -DELTA_LIMIT, DELTA_LIMIT)
        ctrl_new  = np.clip(ctrl  + delta_new, JOINT_LO, JOINT_HI)
        return ctrl_new, delta_new

    def is_done():
        _, p, _ = orientation()
        return mjdata.qpos[2] < 0.03 or abs(p) > math.radians(45)

    def reset_sim():
        ip = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
        mujoco.mj_resetData(mjmodel, mjdata)
        mjdata.qpos[7:15] = ip
        mjdata.qpos[2]    = 0.10
        mujoco.mj_forward(mjmodel, mjdata)
        for _ in range(int(SETTLE_TIME / mjmodel.opt.timestep)):
            mjdata.ctrl[:] = ip
            mujoco.mj_step(mjmodel, mjdata)
        return ip.copy(), np.zeros(ACTION_DIM, dtype=np.float32)

    # ── Rollout ───────────────────────────────────────────────────────────────
    ctrl, delta = reset_sim()
    x0          = mjdata.qpos[0]
    _, _, yaw0  = orientation()
    tot_r       = 0.0
    s           = build_state(delta)

    states, actions, log_probs, rewards, values, dones = [], [], [], [], [], []

    done = False
    for _ in range(int(EPISODE_DUR / CTRL_DT)):
        st = torch.FloatTensor(s).unsqueeze(0)
        with torch.no_grad():
            action, log_prob, value = ac.get_action(st)
        an = action.squeeze(0).numpy()
        lp = log_prob.item()
        v  = value.item()

        ctrl, delta = apply_ddelta(ctrl, delta, an)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)

        roll, pitch, yaw = orientation()
        dyaw     = (yaw - yaw0 + math.pi) % (2 * math.pi) - math.pi
        vx       = float(mjdata.qvel[0])
        vy       = abs(float(mjdata.qvel[1]))
        jvel_mag = float(np.mean(np.abs(mjdata.qvel[6:14])))
        jpos     = mjdata.qpos[7:15].astype(np.float32)
        excursion = float(np.mean(np.abs(jpos - OFFSETS)))

        vx_err = max(0.0, abs(vx - TARGET_VX) - VX_TOL)
        r = (-VX_TRACK_W * vx_err**2
             + ALIVE_BONUS
             - 0.10 * vy
             - ACTION_REG_W * float(np.sum(an**2))
             - DELTA_REG_W  * float(np.sum(delta**2))
             + JVEL_W       * jvel_mag
             + JEXC_W       * excursion
             - ALIGN_W      * (roll**2 + pitch**2)
             - YAW_W        * dyaw**2)
        tot_r += r
        done = is_done()
        if done:
            r -= 5.0

        states.append(s); actions.append(an)
        log_probs.append(lp); rewards.append(r)
        values.append(v); dones.append(float(done))
        s = build_state(delta)
        if done:
            break

    if done:
        last_value = 0.0
    else:
        with torch.no_grad():
            last_value = ac.vf(torch.FloatTensor(s).unsqueeze(0)).squeeze().item()

    return {
        "states":     states,
        "actions":    actions,
        "log_probs":  log_probs,
        "rewards":    rewards,
        "values":     values,
        "dones":      dones,
        "last_value": last_value,
        "dist":       mjdata.qpos[0] - x0,
        "tot_r":      tot_r,
    }
'''

with open('_parallel_worker.py', 'w') as f:
    f.write(_WORKER_CODE)
print('Worker module written to _parallel_worker.py')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mujoco
import mediapy as media
import matplotlib.pyplot as plt
from tqdm import trange

# Use spawn context — required on macOS because MuJoCo is not fork-safe.
# (On macOS, Python 3.8+ already defaults to spawn, but we set it explicitly.)
import multiprocessing
multiprocessing.set_start_method('spawn', force=True)
from concurrent.futures import ProcessPoolExecutor

os.makedirs('movies',       exist_ok=True)
os.makedirs('models',       exist_ok=True)
os.makedirs('plots',        exist_ok=True)

# ── Hyper-parameters (same as AllLegs_def) ────────────────────────────────────
N_WORKERS    = 4             # number of parallel MuJoCo processes (~4× speedup)
N_ROUNDS     = 2500          # training rounds (1 round = N_WORKERS episodes)

ALIGN_W      = 0.00
YAW_W        = 0.20
ALIVE_BONUS  = 0.05

MAX_DDELTA   = 0.02
DELTA_LIMIT  = 0.05

ACTION_REG_W = 0.0001
DELTA_REG_W  = 0.0002
JVEL_W       = 0.05
JEXC_W       = 0.6

TARGET_VX    = 1.0 / 4.0    # 0.25 m/s
VX_TRACK_W   = 4.0
VX_TOL       = 0.02

# ── PPO ───────────────────────────────────────────────────────────────────────
LR                 = 3e-4
GAMMA              = 0.99
GAE_LAMBDA         = 0.95
CLIP_EPS           = 0.2
ENTROPY_COEF       = 0.005
VF_COEF            = 0.5
MAX_GRAD           = 0.5
N_STEPS_PER_UPDATE = 4096
PPO_EPOCHS         = 10
MINIBATCH_SIZE     = 256

LOG_STD_INIT = 0.5
LOG_STD_MIN  = -2.0
LOG_STD_MAX  = 1.0

RENDER_EVERY = 50            # render every N rounds (more frequent since rounds are bigger)
LOAD_MODEL   = ''            # e.g. 'models/parallel_best.pth'

# ── Dimensions ────────────────────────────────────────────────────────────────
ACTION_DIM = 8
HIDDEN_DIM = 64
STATE_DIM  = 24

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}   |   N_WORKERS: {N_WORKERS}')

In [ ]:
# ── Robot geometry + MuJoCo (main-process instance, used for eval only) ───────
HIP_INIT  =  0.90
KNEE_INIT = -1.40
DELTA_HIP  = 0.50
DELTA_KNEE = 0.30
HIP_LO,  HIP_HI  = HIP_INIT - DELTA_HIP,  HIP_INIT + DELTA_HIP
KNEE_LO, KNEE_HI = KNEE_INIT - DELTA_KNEE, KNEE_INIT + DELTA_KNEE
OFFSETS   = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
JOINT_LO  = np.array([HIP_LO, KNEE_LO] * 4, dtype=np.float32)
JOINT_HI  = np.array([HIP_HI, KNEE_HI] * 4, dtype=np.float32)

SUBSTEPS    = 5
SETTLE_TIME = 0.3
EPISODE_DUR = 8.0
XML_PATH    = os.path.abspath('dog.xml')  # absolute path so subprocesses can find it

mjmodel = mujoco.MjModel.from_xml_path(XML_PATH)
mjdata  = mujoco.MjData(mjmodel)
CTRL_DT = mjmodel.opt.timestep * SUBSTEPS
print(f'CTRL_DT = {CTRL_DT*1000:.1f} ms  |  episode = {EPISODE_DUR}s = {int(EPISODE_DUR/CTRL_DT)} steps')
print(f'N_WORKERS={N_WORKERS} → {N_WORKERS*int(EPISODE_DUR/CTRL_DT)} steps/round  '
      f'(PPO update every {N_STEPS_PER_UPDATE} steps)')

# ── Config dict passed to every worker ───────────────────────────────────────
# All hyperparameters in one serialisable dict (no tensors, no MuJoCo objects).
CFG = dict(
    STATE_DIM=STATE_DIM, ACTION_DIM=ACTION_DIM, HIDDEN_DIM=HIDDEN_DIM,
    LOG_STD_INIT=LOG_STD_INIT, LOG_STD_MIN=LOG_STD_MIN, LOG_STD_MAX=LOG_STD_MAX,
    SUBSTEPS=SUBSTEPS, SETTLE_TIME=SETTLE_TIME, EPISODE_DUR=EPISODE_DUR,
    XML_PATH=XML_PATH,
    HIP_INIT=HIP_INIT, KNEE_INIT=KNEE_INIT,
    DELTA_HIP=DELTA_HIP, DELTA_KNEE=DELTA_KNEE,
    MAX_DDELTA=MAX_DDELTA, DELTA_LIMIT=DELTA_LIMIT,
    TARGET_VX=TARGET_VX, VX_TRACK_W=VX_TRACK_W, VX_TOL=VX_TOL,
    ALIVE_BONUS=ALIVE_BONUS, JVEL_W=JVEL_W, JEXC_W=JEXC_W,
    ACTION_REG_W=ACTION_REG_W, DELTA_REG_W=DELTA_REG_W,
    ALIGN_W=ALIGN_W, YAW_W=YAW_W,
)

In [ ]:
# ── ActorCritic (main process — used for the PPO update and eval) ─────────────
# The same architecture is reconstructed inside each worker from the weights dict.

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.pi = nn.Sequential(
            nn.Linear(STATE_DIM,  HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, ACTION_DIM),
        )
        nn.init.orthogonal_(self.pi[-1].weight, gain=0.01)
        nn.init.zeros_(self.pi[-1].bias)

        self.log_std = nn.Parameter(torch.full((ACTION_DIM,), LOG_STD_INIT))

        self.vf = nn.Sequential(
            nn.Linear(STATE_DIM, 256), nn.ReLU(),
            nn.Linear(256, 256),       nn.ReLU(),
            nn.Linear(256, 1),
        )
        nn.init.orthogonal_(self.vf[-1].weight, gain=1.0)
        nn.init.zeros_(self.vf[-1].bias)

    def forward_pi(self, s):
        mean    = self.pi(s)
        log_std = self.log_std.clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    def forward_vf(self, s):
        return self.vf(s).squeeze(-1)

    def get_action(self, s, deterministic=False):
        mean, log_std = self.forward_pi(s)
        std = log_std.exp()
        raw = mean if deterministic else mean + std * torch.randn_like(std)
        action = torch.tanh(raw)
        log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                    - log_std - 0.5 * math.log(2 * math.pi))
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob  = log_prob.sum(-1)
        value = self.forward_vf(s)
        return action, log_prob, value

    def evaluate_actions(self, s, actions_tanh):
        mean, log_std = self.forward_pi(s)
        std = log_std.exp()
        raw = torch.atanh(actions_tanh.clamp(-0.999, 0.999))
        log_prob = (-0.5 * ((raw - mean) / (std + 1e-8))**2
                    - log_std - 0.5 * math.log(2 * math.pi))
        log_prob -= torch.log(1 - actions_tanh.pow(2) + 1e-6)
        log_prob  = log_prob.sum(-1)
        entropy = (log_std + 0.5 * math.log(2 * math.pi * math.e)).sum(-1)
        value   = self.forward_vf(s)
        return log_prob, entropy, value

    def weights_numpy(self):
        """Export state_dict as {name: ndarray} for subprocess transmission."""
        return {k: v.cpu().numpy() for k, v in self.state_dict().items()}


ac  = ActorCritic().to(device)
opt = torch.optim.Adam(ac.parameters(), lr=LR, eps=1e-5)

if LOAD_MODEL:
    ck = torch.load(LOAD_MODEL, map_location=device)
    ac.load_state_dict(ck.get('best_w') or ck['ac'])
    if 'opt' in ck: opt.load_state_dict(ck['opt'])
    best_dist = ck.get('best_dist', -np.inf)
    start_rnd = ck.get('rnd', 0) + 1
    rl, dl, cll, all_ = ck.get('rl',[]), ck.get('dl',[]), ck.get('cll',[]), ck.get('all_',[])
    print(f'Loaded {LOAD_MODEL}  best={best_dist:.3f}m  round={start_rnd}')
else:
    best_dist = -np.inf
    start_rnd = 0
    rl, dl, cll, all_ = [], [], [], []

best_w = None
print(f'ActorCritic params: {sum(p.numel() for p in ac.parameters()):,}')

In [ ]:
# ── PPO buffer + GAE + PPO update ─────────────────────────────────────────────

class PPOBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states    = []
        self.actions   = []
        self.log_probs = []
        self.rewards   = []
        self.values    = []
        self.dones     = []

    def push_episode(self, result: dict):
        """Append one worker's result dict into the buffer."""
        self.states.extend(result['states'])
        self.actions.extend(result['actions'])
        self.log_probs.extend(result['log_probs'])
        self.rewards.extend(result['rewards'])
        self.values.extend(result['values'])
        self.dones.extend(result['dones'])

    def __len__(self):
        return len(self.states)

    def compute_gae(self, last_values: list):
        """
        GAE over the full buffer.
        last_values: list of bootstrap values, one per episode in order they were appended.
        We treat each episode boundary (done=1) as a terminal, and handle bootstrap
        at the true episode end via the stored dones.
        For simplicity we compute a single GAE pass over the concatenated buffer
        using dones to reset the advantage at episode boundaries — standard practice.
        The final last_value used for bootstrapping is the mean of worker last_values.
        """
        T       = len(self.rewards)
        adv     = np.zeros(T, dtype=np.float32)
        ret     = np.zeros(T, dtype=np.float32)
        last_v  = np.mean(last_values)  # average bootstrap across workers
        values  = np.array(self.values + [last_v], dtype=np.float32)
        rewards = np.array(self.rewards, dtype=np.float32)
        dones   = np.array(self.dones,   dtype=np.float32)
        gae = 0.0
        for t in reversed(range(T)):
            delta  = rewards[t] + GAMMA * values[t+1] * (1-dones[t]) - values[t]
            gae    = delta + GAMMA * GAE_LAMBDA * (1-dones[t]) * gae
            adv[t] = gae
            ret[t] = adv[t] + values[t]
        return ret, adv

    def get_tensors(self, last_values):
        ret, adv = self.compute_gae(last_values)
        S   = torch.FloatTensor(np.array(self.states)).to(device)
        A   = torch.FloatTensor(np.array(self.actions)).to(device)
        LP  = torch.FloatTensor(np.array(self.log_probs)).to(device)
        R   = torch.FloatTensor(ret).to(device)
        Adv = torch.FloatTensor(adv).to(device)
        Adv = (Adv - Adv.mean()) / (Adv.std() + 1e-8)
        return S, A, LP, R, Adv


def update_ppo(buf: PPOBuffer, last_values: list):
    S, A, LP_old, R, Adv = buf.get_tensors(last_values)
    N = S.shape[0]
    total_pi = total_vf = n_up = 0
    for _ in range(PPO_EPOCHS):
        idx = torch.randperm(N, device=device)
        for start in range(0, N, MINIBATCH_SIZE):
            mb = idx[start:start+MINIBATCH_SIZE]
            lp_new, entropy, v_new = ac.evaluate_actions(S[mb], A[mb])
            ratio  = (lp_new - LP_old[mb]).exp()
            surr1  = ratio * Adv[mb]
            surr2  = ratio.clamp(1-CLIP_EPS, 1+CLIP_EPS) * Adv[mb]
            pi_loss = -torch.min(surr1, surr2).mean()
            vf_loss = F.mse_loss(v_new, R[mb])
            loss = pi_loss + VF_COEF * vf_loss - ENTROPY_COEF * entropy.mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(ac.parameters(), MAX_GRAD)
            opt.step()
            total_pi += pi_loss.item()
            total_vf += vf_loss.item()
            n_up     += 1
    buf.clear()
    return total_vf / max(n_up,1), total_pi / max(n_up,1)


ppo_buf = PPOBuffer()

In [ ]:
# ── Eval rollout (runs in main process, deterministic, with optional render) ───

def orientation():
    qw, qx, qy, qz = mjdata.qpos[3:7]
    roll  = math.atan2(2*(qw*qx + qy*qz), 1 - 2*(qx**2 + qy**2))
    pitch = math.asin(float(np.clip(2*(qw*qy - qz*qx), -1, 1)))
    yaw   = math.atan2(2*(qw*qz + qx*qy), 1 - 2*(qy**2 + qz**2))
    return roll, pitch, yaw


def build_state(delta):
    return np.concatenate([
        mjdata.qpos[7:15].astype(np.float32),
        mjdata.qvel[6:14].astype(np.float32),
        delta.astype(np.float32),
    ])


def apply_ddelta(ctrl, delta, ddelta):
    delta_new = np.clip(delta + ddelta * MAX_DDELTA, -DELTA_LIMIT, DELTA_LIMIT)
    ctrl_new  = np.clip(ctrl  + delta_new, JOINT_LO, JOINT_HI)
    return ctrl_new, delta_new


def reset_sim():
    ip = np.array([HIP_INIT, KNEE_INIT] * 4, dtype=np.float32)
    mujoco.mj_resetData(mjmodel, mjdata)
    mjdata.qpos[7:15] = ip
    mjdata.qpos[2]    = 0.10
    mujoco.mj_forward(mjmodel, mjdata)
    for _ in range(int(SETTLE_TIME / mjmodel.opt.timestep)):
        mjdata.ctrl[:] = ip
        mujoco.mj_step(mjmodel, mjdata)
    return ip.copy(), np.zeros(ACTION_DIM, dtype=np.float32)


def rollout_eval(render=False, fps=60):
    ctrl, delta = reset_sim()
    x0     = mjdata.qpos[0]
    frames = []
    renderer = None

    if render:
        renderer = mujoco.Renderer(mjmodel, height=720, width=1280)
        cam = mujoco.MjvCamera()
        mujoco.mjv_defaultFreeCamera(mjmodel, cam)
        cam.distance  = 0.6
        cam.elevation = -20
        fe = max(1, int(1 / (fps * CTRL_DT)))

    s = build_state(delta)
    for t in range(int(EPISODE_DUR / CTRL_DT)):
        with torch.no_grad():
            st = torch.FloatTensor(s).unsqueeze(0).to(device)
            action, _, _ = ac.get_action(st, deterministic=True)
            an = action.squeeze(0).cpu().numpy()
        ctrl, delta = apply_ddelta(ctrl, delta, an)
        mjdata.ctrl[:] = ctrl
        for _ in range(SUBSTEPS):
            mujoco.mj_step(mjmodel, mjdata)
        s = build_state(delta)
        if render and t % fe == 0:
            cam.lookat[0] = mjdata.qpos[0]
            cam.lookat[1] = mjdata.qpos[1]
            cam.azimuth   = (cam.azimuth + 0.2) % 360
            renderer.update_scene(mjdata, cam)
            frames.append(renderer.render().copy())
        _, p, _ = orientation()
        if mjdata.qpos[2] < 0.03 or abs(p) > math.radians(45):
            break

    dist = mjdata.qpos[0] - x0
    if renderer: renderer.close()
    return dist, frames

In [ ]:
# ── Parallel training loop ────────────────────────────────────────────────────
#
# Each round:
#   1. Serialize current weights to numpy (cheap, avoids pickle overhead of tensors)
#   2. Submit N_WORKERS futures — each runs one full episode in a subprocess
#   3. Collect results, push into shared PPOBuffer
#   4. When buffer >= N_STEPS_PER_UPDATE, run PPO update
#   5. Periodically save checkpoint and render eval video

from _parallel_worker import _worker

nm = 0
ppo_update_count = 0
t0 = time.time()

# Keep track of last_values across the batch for proper GAE bootstrap
pending_last_values = []

with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
    for rnd in trange(start_rnd, start_rnd + N_ROUNDS, desc='rounds'):

        # ── 1. Snapshot current weights as numpy (pickleable) ──────────────
        w_np = ac.weights_numpy()

        # ── 2. Launch N_WORKERS parallel episodes ───────────────────────────
        futures = [
            pool.submit(_worker, wid, w_np, CFG)
            for wid in range(N_WORKERS)
        ]

        # ── 3. Collect results + fill buffer ────────────────────────────────
        round_dists = []
        round_rs    = []
        last_values = []

        for fut in futures:
            res = fut.result()          # blocks until this worker is done
            ppo_buf.push_episode(res)
            last_values.append(res['last_value'])
            round_dists.append(res['dist'])
            round_rs.append(res['tot_r'])

        # Keep track of bootstrap values; they line up with the buffer segments
        pending_last_values.extend(last_values)

        avg_dist = float(np.mean(round_dists))
        avg_r    = float(np.mean(round_rs))
        dl.append(avg_dist)
        rl.append(avg_r)

        if avg_dist > best_dist:
            best_dist = avg_dist
            best_w = {k: v.clone() for k, v in ac.state_dict().items()}

        # ── 4. PPO update when buffer is full ────────────────────────────────
        cl = al = 0.0
        if len(ppo_buf) >= N_STEPS_PER_UPDATE:
            cl, al = update_ppo(ppo_buf, pending_last_values)
            pending_last_values = []
            ppo_update_count += 1
        cll.append(cl); all_.append(al)

        # ── 5. Logging ───────────────────────────────────────────────────────
        if rnd % 5 == 0:
            elapsed = time.time() - t0
            steps_per_sec = (rnd - start_rnd + 1) * N_WORKERS * int(EPISODE_DUR/CTRL_DT) / elapsed
            print(f'  rnd {rnd:4d} | dist={avg_dist:.3f}m avg10={np.mean(dl[-10:]):.3f} '
                  f'| R={avg_r:.1f} | buf={len(ppo_buf)} '
                  f'| {steps_per_sec:.0f} steps/s | updates={ppo_update_count}')

        # ── 6. Periodic eval render ──────────────────────────────────────────
        if rnd % RENDER_EVERY == 0 and rnd > start_rnd and best_w:
            train_w = {k: v.clone() for k, v in ac.state_dict().items()}
            ac.load_state_dict(best_w)
            bd, frames = rollout_eval(render=True)
            if frames:
                out = f'movies/parallel_ppo_{nm:03d}_rnd{rnd:04d}_{bd:.2f}m.mp4'
                media.write_video(out, frames, fps=60)
                print(f'  → {out}')
            nm += 1
            ac.load_state_dict(train_w)

        # ── 7. Checkpoint every 50 rounds ────────────────────────────────────
        if rnd % 50 == 49:
            torch.save({
                'ac': ac.state_dict(), 'opt': opt.state_dict(),
                'best_w': best_w, 'best_dist': best_dist,
                'rnd': rnd, 'rl': rl, 'dl': dl, 'cll': cll, 'all_': all_,
            }, f'models/parallel_ppo_rnd{rnd}.pth')

print(f'\nDone. best_dist={best_dist:.3f}m  |  {ppo_update_count} PPO updates')

In [ ]:
# ── Results ───────────────────────────────────────────────────────────────────
w = 20
sm = lambda x: np.convolve(x, np.ones(w)/w, 'valid')

if len(dl) >= w:
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    ax[0,0].plot(sm(dl),   color='#2ecc71'); ax[0,0].set_title('Distance (m)')
    ax[0,1].plot(sm(rl),   color='#3498db'); ax[0,1].set_title('Reward')
    ax[1,0].plot(sm(cll),  color='#e74c3c'); ax[1,0].set_title('Value Loss')
    ax[1,1].plot(sm(all_), color='#f39c12'); ax[1,1].set_title('Policy Loss')
    for a_ in ax.flat:
        a_.grid(True, alpha=0.3); a_.set_xlabel('round')
    plt.suptitle(f'Parallel PPO ({N_WORKERS} workers) | best={best_dist:.3f}m')
    plt.tight_layout()
    plt.savefig('plots/parallel_ppo_results.png', dpi=120)
    plt.show()

# Final eval video
if best_w:
    ac.load_state_dict(best_w)
    bd, frames = rollout_eval(render=True)
    if frames:
        media.write_video('movies/parallel_ppo_best.mp4', frames, fps=60)
        print(f'Saved movies/parallel_ppo_best.mp4  dist={bd:.3f}m')
        media.show_video(frames, fps=60)